# Solutions – Day 22 Exercises

In [ ]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from PIL import Image
import cv2
import numpy as np
import requests

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load base pipeline (reused)
controlnet_canny = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet_canny, torch_dtype=torch.float16
).to(device)
pipe.safety_checker = None

## Exercise 1: Sketch from real photo

In [ ]:
url = "https://images.dog.ceo/breeds/hound-afghan/n02088094_1003.jpg"
photo = Image.open(requests.get(url, stream=True).raw).convert("RGB").resize((512,512))
# Canny edges
photo_gray = np.array(photo.convert("L"))
edges = cv2.Canny(photo_gray, 50, 150)
control_img = Image.fromarray(edges)

prompt = "a beautiful dog, detailed"
image_with_control = pipe(prompt, image=control_img, num_inference_steps=20).images[0]
image_without_control = pipe(prompt, num_inference_steps=20).images[0]

# Display both
image_with_control

## Exercise 2: Different ControlNet type (scribble)

In [ ]:
controlnet_scribble = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-scribble", torch_dtype=torch.float16)
pipe_scribble = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet_scribble, torch_dtype=torch.float16
).to(device)
pipe_scribble.safety_checker = None

# Use same sketch as in notebook
output = pipe_scribble(prompt="cat", image=control_img, num_inference_steps=20).images[0]
output

## Exercise 3: Control strength sweep

In [ ]:
for scale in [0.2, 0.7, 1.2]:
    img = pipe(prompt="cat", image=control_img, controlnet_conditioning_scale=scale, num_inference_steps=20).images[0]
    print(f"Scale {scale}: output varies; low scale ignores edges, high scale may cause artifacts.")

## Exercise 4: Hand‑drawn sketch
Load a photo of your hand‑drawn sketch, preprocess with `cv2.Canny` or invert colors. Then run pipeline as above.

In [ ]:
# hand_sketch = Image.open("my_sketch.jpg").convert("L")
# edges = cv2.Canny(np.array(hand_sketch), 50, 150)
# control = Image.fromarray(edges)
# result = pipe(prompt="a castle", image=control).images[0]

## Exercise 5: IP‑Adapter (conceptual solution)
"""
Install IP-Adapter:
git clone https://github.com/tencent-ailab/IP-Adapter
cd IP-Adapter
pip install -r requirements.txt

Then run:
from ip_adapter import IPAdapter
sd_pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
ip_model = IPAdapter(sd_pipe, "ip-adapter_sd15.bin", device="cuda")
ref_img = Image.open("van_gogh_style.jpg")
images = ip_model.generate(prompt="a cat", image=ref_img, scale=0.7)
To combine with ControlNet, use a custom pipeline – advanced.
"""